# Multimodal Contradiction Project

This notebook implements the project pipeline in short sections.
The helper modules under `src/vl_contradiction` hold the heavier logic so each cell stays readable.

## 1. Notebook Bootstrap

Use this section to resolve the project root, optionally install dependencies, and expose the package for imports.
When running in Colab, the notebook clones the repo into `/content/project`, keeps raw COCO files on local Colab disk, and writes smaller artifacts to Google Drive.

In [ ]:
import base64
from pathlib import Path
import shutil
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import userdata

    GITHUB_USERNAME = "sajpatel15"
    GITHUB_REPO = "comp_646_project"
    PROJECT_ROOT = Path("/content/project")
    token = userdata.get("GITHUB_TOKEN")
    if not token:
        raise ValueError("Missing GITHUB_TOKEN Colab secret.")
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()

    if PROJECT_ROOT.exists() and not (PROJECT_ROOT / "pyproject.toml").exists():
        shutil.rmtree(PROJECT_ROOT)

    if (PROJECT_ROOT / ".git").exists():
        pull_result = subprocess.run(
            [
                "git",
                "-c",
                f"http.extraheader=AUTHORIZATION: basic {auth}",
                "-C",
                str(PROJECT_ROOT),
                "pull",
                "--ff-only",
            ],
            capture_output=True,
            text=True,
        )
        print(pull_result.stdout)
        if pull_result.returncode != 0:
            print(pull_result.stderr)
            raise RuntimeError("Git pull failed. Remove /content/project or fix the repo state.")

    if not (PROJECT_ROOT / "pyproject.toml").exists():
        repo_url = f"https://github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git"
        clone_result = subprocess.run(
            [
                "git",
                "-c",
                f"http.extraheader=AUTHORIZATION: basic {auth}",
                "clone",
                repo_url,
                str(PROJECT_ROOT),
            ],
            capture_output=True,
            text=True,
        )
        print(clone_result.stdout)
        if clone_result.returncode != 0:
            print(clone_result.stderr)
            raise RuntimeError("Git clone failed. Check GITHUB_TOKEN permissions for the repo.")
else:
    PROJECT_ROOT = Path.cwd().resolve()
    if PROJECT_ROOT.name == "notebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent
    if not (PROJECT_ROOT / "pyproject.toml").exists():
        raise FileNotFoundError(f"Could not locate project root from {PROJECT_ROOT}")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_ROOT)], check=True)

src_path = PROJECT_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f"Project root: {PROJECT_ROOT}")
print(f"Project files found: {(PROJECT_ROOT / 'pyproject.toml').exists()}")

In [ ]:
print(f"Project root exists: {PROJECT_ROOT.exists()}")
print(f"Package root exists: {(PROJECT_ROOT / 'src' / 'vl_contradiction').exists()}")
sorted(path.name for path in (PROJECT_ROOT / 'src').iterdir())

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from vl_contradiction.audit import build_audit_sheet, summarize_audit
from vl_contradiction.benchmark import build_benchmark, sample_qwen_subset
from vl_contradiction.coco import ensure_coco_dataset, load_coco_caption_context
from vl_contradiction.config import load_config
from vl_contradiction.metrics import (
    bootstrap_macro_f1_ci,
    compute_classification_metrics,
    expected_calibration_error,
    fit_temperature,
)
from vl_contradiction.runtime import detect_runtime, ensure_directories, print_runtime_summary, set_global_seed

In [ ]:
from vl_contradiction.clip_baselines import (
    CLASS_ORDER,
    compute_similarity_scores,
    extract_joint_features,
    extract_token_features,
    fit_similarity_thresholds,
    load_clip_bundle,
    predict_with_thresholds,
)
from vl_contradiction.models import CrossAttentionFusionClassifier, LinearProbe
from vl_contradiction.plotting import (
    save_bar_chart,
    save_confusion_matrix,
    save_qualitative_panel,
    save_reliability_diagram,
    save_score_histogram,
    save_threshold_sweep,
    save_training_curves,
)
from vl_contradiction.qwen import load_qwen_bundle, run_qwen_inference
from vl_contradiction.training import FeatureDataset, TokenDataset, create_loader, evaluate_model, train_model


In [ ]:
from IPython import get_ipython
from PIL import Image


def show_saved_figure(path, title=None, figsize=(8, 5)):
    path = Path(path)
    if not path.exists():
        print(f"Missing figure: {path}")
        return
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(Image.open(path))
    ax.axis("off")
    if title:
        ax.set_title(title)
    plt.tight_layout()
    plt.show()
    plt.close(fig)


def show_prediction_examples(frame, title, n=4, seed=42):
    if frame.empty:
        print(f"No rows available for {title}")
        return

    sample = frame.sample(min(n, len(frame)), random_state=seed).reset_index(drop=True)
    fig, axes = plt.subplots(len(sample), 2, figsize=(12, 4 * len(sample)))
    if len(sample) == 1:
        axes = np.array([axes])

    for axis_row, (_, row) in zip(axes, sample.iterrows(), strict=True):
        image = Image.open(row["file_path"]).convert("RGB")
        axis_row[0].imshow(image)
        axis_row[0].axis("off")

        lines = [
            f"True: {row['label']}",
            f"Edit family: {row['edit_family']}",
            f"Source: {row['source_caption']}",
            f"Edited: {row['edited_caption']}",
        ]
        if "pred_label" in frame.columns:
            lines.insert(1, f"Pred: {row['pred_label']}")
        if "raw_score" in frame.columns and pd.notna(row.get("raw_score")):
            lines.append(f"Raw score: {float(row['raw_score']):.4f}")
        if "confidence" in frame.columns and pd.notna(row.get("confidence")):
            lines.append(f"Confidence: {float(row['confidence']):.4f}")

        axis_row[1].axis("off")
        axis_row[1].text(0.0, 1.0, "\n".join(lines), va="top", fontsize=10)

    fig.suptitle(title)
    fig.tight_layout()
    plt.show()
    plt.close(fig)

## 2. Configuration and Runtime

Load the shared config, resolve runtime paths, and create the artifact directories up front.

In [ ]:
config = load_config(PROJECT_ROOT / "configs" / "default.yaml")
runtime = detect_runtime(PROJECT_ROOT, config)
ensure_directories(
    [
        runtime.cache_root,
        runtime.dataset_root,
        runtime.benchmark_root,
        runtime.checkpoint_root,
        runtime.log_root,
        runtime.metrics_root,
        runtime.figure_root,
        runtime.qwen_root,
    ]
)
set_global_seed(config.runtime.seed)
print_runtime_summary(runtime)
print(f"Image splits for this run: {config.data.image_splits}")

In [ ]:
RUN_RAW_CLIP = True
RUN_LINEAR_PROBE = True
RUN_CROSS_ATTENTION = True
RUN_QWEN = False
CURRENT_STAGE = "prototype"

family_limit = getattr(config.data, f"{CURRENT_STAGE}_families")
print(f"Stage: {CURRENT_STAGE} | families: {family_limit}")

## 2A. Live Monitoring

Use this optional cell to open TensorBoard inline while training logs are being written.

In [ ]:
ip = get_ipython()
if ip is None:
    print(f"TensorBoard logs will be written to: {runtime.log_root}")
else:
    try:
        ip.run_line_magic("load_ext", "tensorboard")
    except Exception:
        pass
    print(f"TensorBoard logdir: {runtime.log_root}")
    try:
        ip.run_line_magic("tensorboard", f"--logdir {runtime.log_root}")
    except Exception as exc:
        print(f"TensorBoard launch note: {exc}")

## 3. COCO Download and Validation

This section checks the local Colab dataset cache first and downloads only the requested COCO image splits plus annotations when needed.

In [ ]:
coco_paths = ensure_coco_dataset(runtime.dataset_root, download=True, image_splits=config.data.image_splits)
print(coco_paths)

In [ ]:
coco_frame = load_coco_caption_context(coco_paths, splits=config.data.image_splits)
print(coco_frame.shape)
coco_frame.head(2)

## 4. Benchmark Construction

Build the benchmark from deterministic edits, then save the records and the family split manifest.

In [ ]:
benchmark_result = build_benchmark(
    coco_frame=coco_frame,
    family_limit=family_limit,
    split_ratio=config.data.split_ratio,
    seed=config.runtime.split_seed,
)
benchmark_frame = benchmark_result.records.copy()
family_manifest = benchmark_result.family_manifest.copy()

benchmark_path = runtime.benchmark_root / f"benchmark_{CURRENT_STAGE}.csv"
manifest_path = runtime.benchmark_root / f"family_manifest_{CURRENT_STAGE}.csv"
benchmark_frame.to_csv(benchmark_path, index=False)
family_manifest.to_csv(manifest_path, index=False)

if benchmark_frame.empty:
    raise RuntimeError("Benchmark generation returned zero rows. Check the COCO split and edit rules.")

print(f"Saved benchmark to {benchmark_path}")
print(f"Saved family manifest to {manifest_path}")
print(benchmark_frame[["label", "edit_family", "split"]].value_counts().head(12))
show_prediction_examples(benchmark_frame, "Benchmark Spot Check", n=4, seed=config.runtime.seed)
benchmark_frame.head(3)

### Manual Audit Note

Fill in `label_valid` and `grammar_ok` in the generated audit CSV, then rerun the next cell.


In [ ]:
audit_sheet = build_audit_sheet(
    benchmark_frame,
    per_family=config.data.audit_samples_per_family,
    seed=config.runtime.seed,
)
audit_path = runtime.benchmark_root / f"audit_sheet_{CURRENT_STAGE}.csv"
audit_sheet.to_csv(audit_path, index=False)
print(f"Saved audit sheet to {audit_path}")
audit_sheet.head(10)

## 5. Split Views

Keep split-specific frames small and explicit so downstream sections stay easy to read.

In [ ]:
split_frames = {
    split: benchmark_frame.loc[benchmark_frame['split'] == split].reset_index(drop=True)
    for split in ['train', 'val', 'test']
}
for split, frame in split_frames.items():
    print(f"{split}: {frame.shape}")

## 6. Raw CLIP Baseline

Fit CLIP thresholds on validation only, then evaluate on test.

In [ ]:
clip_bundle = load_clip_bundle(config.model.clip_name, runtime.device)

In [ ]:
if RUN_RAW_CLIP:
    val_scores = compute_similarity_scores(split_frames['val'], clip_bundle, batch_size=config.training.batch_size)
    threshold_config, threshold_search = fit_similarity_thresholds(
        labels=val_scores['label'].tolist(),
        scores=val_scores['raw_score'].to_numpy(),
        grid_size=config.evaluation.threshold_grid_size,
    )
    test_scores = compute_similarity_scores(split_frames['test'], clip_bundle, batch_size=config.training.batch_size)
    test_predictions = predict_with_thresholds(
        test_scores['raw_score'].to_numpy(),
        tau_low=threshold_config['tau_low'],
        tau_high=threshold_config['tau_high'],
    )
    y_true = np.array([CLASS_ORDER.index(label) for label in test_scores['label']])
    clip_metrics = compute_classification_metrics(y_true, test_predictions)
    clip_ci = bootstrap_macro_f1_ci(y_true, test_predictions, samples=config.evaluation.bootstrap_samples, seed=config.runtime.seed)
    print("Raw CLIP metrics", clip_metrics)
    print("Raw CLIP macro-F1 95% CI", clip_ci)

In [ ]:
if RUN_RAW_CLIP:
    raw_clip_view = split_frames['test'].merge(
        test_scores[['sample_id', 'raw_score']],
        on='sample_id',
        how='left',
    ).copy()
    raw_clip_view['pred_label'] = [CLASS_ORDER[idx] for idx in test_predictions]
    raw_clip_view['correct'] = raw_clip_view['label'] == raw_clip_view['pred_label']

    clip_score_pdf = runtime.figure_root / f"clip_scores_{CURRENT_STAGE}.pdf"
    clip_score_png = runtime.figure_root / f"clip_scores_{CURRENT_STAGE}.png"
    clip_sweep_pdf = runtime.figure_root / f"clip_threshold_sweep_{CURRENT_STAGE}.pdf"
    clip_sweep_png = runtime.figure_root / f"clip_threshold_sweep_{CURRENT_STAGE}.png"
    clip_conf_pdf = runtime.figure_root / f"clip_confusion_{CURRENT_STAGE}.pdf"
    clip_conf_png = runtime.figure_root / f"clip_confusion_{CURRENT_STAGE}.png"

    for path in [clip_score_pdf, clip_score_png]:
        save_score_histogram(test_scores, path, "Raw CLIP Score Distribution")
    for path in [clip_sweep_pdf, clip_sweep_png]:
        save_threshold_sweep(threshold_search, path, "Validation Threshold Sweep")
    for path in [clip_conf_pdf, clip_conf_png]:
        save_confusion_matrix(clip_metrics['confusion_matrix'], CLASS_ORDER, path, "Raw CLIP Confusion Matrix")

    raw_clip_metrics_path = runtime.metrics_root / f"raw_clip_metrics_{CURRENT_STAGE}.json"
    raw_clip_metrics_path.write_text(json.dumps({**clip_metrics, 'macro_f1_ci': clip_ci, **threshold_config}, indent=2), encoding='utf-8')
    print(f"Saved raw CLIP metrics to {raw_clip_metrics_path}")
    print(f"Thresholds: tau_low={threshold_config['tau_low']:.4f}, tau_high={threshold_config['tau_high']:.4f}")

    show_saved_figure(clip_score_png, "Raw CLIP Score Distribution")
    show_saved_figure(clip_sweep_png, "Raw CLIP Threshold Sweep")
    show_saved_figure(clip_conf_png, "Raw CLIP Confusion Matrix")
    show_prediction_examples(raw_clip_view.query('correct'), "Raw CLIP Correct Examples", n=3, seed=7)
    show_prediction_examples(raw_clip_view.query('not correct'), "Raw CLIP Failure Examples", n=3, seed=11)

## 7. Frozen CLIP Linear Probe

Extract frozen features once, then train a small probe with per-epoch logging.

In [ ]:
if RUN_LINEAR_PROBE:
    feature_store = {}
    label_store = {}
    for split in ['train', 'val', 'test']:
        features, labels = extract_joint_features(split_frames[split], clip_bundle, batch_size=config.training.batch_size)
        feature_store[split] = features
        label_store[split] = labels
        print(f"Prepared {split} features: {tuple(features.shape)}")

In [ ]:
if RUN_LINEAR_PROBE:
    train_loader = create_loader(
        FeatureDataset(feature_store['train'], label_store['train']),
        batch_size=config.training.batch_size,
        shuffle=True,
        num_workers=config.training.num_workers,
    )
    val_loader = create_loader(
        FeatureDataset(feature_store['val'], label_store['val']),
        batch_size=config.training.batch_size,
        shuffle=False,
        num_workers=config.training.num_workers,
    )
    linear_probe = LinearProbe(input_dim=feature_store['train'].shape[1])
    linear_result = train_model(
        model=linear_probe,
        train_loader=train_loader,
        val_loader=val_loader,
        device=runtime.device,
        epochs=config.training.epochs,
        learning_rate=config.training.learning_rate,
        weight_decay=config.training.weight_decay,
        log_dir=runtime.log_root / f"linear_probe_{CURRENT_STAGE}",
        checkpoint_path=runtime.checkpoint_root / f"linear_probe_{CURRENT_STAGE}.pt",
    )

In [ ]:
if RUN_LINEAR_PROBE:
    test_loader = create_loader(
        FeatureDataset(feature_store['test'], label_store['test']),
        batch_size=config.training.batch_size,
        shuffle=False,
        num_workers=config.training.num_workers,
    )
    linear_metrics, linear_logits, linear_labels = evaluate_model(linear_probe.to(runtime.device), test_loader, runtime.device)
    linear_predictions = linear_logits.argmax(dim=1).numpy()
    linear_confidence = torch.softmax(linear_logits, dim=1).max(dim=1).values.numpy()

    linear_view = split_frames['test'].copy()
    linear_view['pred_label'] = [CLASS_ORDER[idx] for idx in linear_predictions]
    linear_view['confidence'] = linear_confidence
    linear_view['correct'] = linear_view['label'] == linear_view['pred_label']

    linear_train_pdf = runtime.figure_root / f"linear_probe_training_{CURRENT_STAGE}.pdf"
    linear_train_png = runtime.figure_root / f"linear_probe_training_{CURRENT_STAGE}.png"
    linear_conf_pdf = runtime.figure_root / f"linear_probe_confusion_{CURRENT_STAGE}.pdf"
    linear_conf_png = runtime.figure_root / f"linear_probe_confusion_{CURRENT_STAGE}.png"

    for path in [linear_train_pdf, linear_train_png]:
        save_training_curves(linear_result.history, path, "Linear Probe Training")
    for path in [linear_conf_pdf, linear_conf_png]:
        save_confusion_matrix(linear_metrics['confusion_matrix'], CLASS_ORDER, path, "Linear Probe Confusion Matrix")

    (runtime.metrics_root / f"linear_probe_metrics_{CURRENT_STAGE}.json").write_text(json.dumps(linear_metrics, indent=2), encoding='utf-8')
    print("Linear probe metrics", linear_metrics)

    show_saved_figure(linear_train_png, "Linear Probe Training Curves")
    show_saved_figure(linear_conf_png, "Linear Probe Confusion Matrix")
    show_prediction_examples(linear_view.query('correct'), "Linear Probe Correct Examples", n=3, seed=13)
    show_prediction_examples(linear_view.query('not correct'), "Linear Probe Failure Examples", n=3, seed=17)

## 8. Cross-Attention Fusion

This section uses frozen CLIP token states and a lightweight cross-attention head.

In [ ]:
if RUN_CROSS_ATTENTION:
    token_store = {}
    token_labels = {}
    for split in ['train', 'val', 'test']:
        image_tokens, text_tokens, labels = extract_token_features(split_frames[split], clip_bundle, batch_size=max(1, config.training.batch_size // 2))
        token_store[split] = (image_tokens, text_tokens)
        token_labels[split] = labels
        print(f"Prepared {split} token tensors: image={tuple(image_tokens.shape)}, text={tuple(text_tokens.shape)}")

In [ ]:
if RUN_CROSS_ATTENTION:
    cross_attention = CrossAttentionFusionClassifier(
        image_input_dim=token_store['train'][0].shape[-1],
        text_input_dim=token_store['train'][1].shape[-1],
        hidden_dim=config.model.hidden_dim,
        num_heads=config.model.num_attention_heads,
        dropout=config.model.dropout,
    )
    cross_train_loader = create_loader(
        TokenDataset(token_store['train'][0], token_store['train'][1], token_labels['train']),
        batch_size=max(1, config.training.batch_size // 2),
        shuffle=True,
        num_workers=config.training.num_workers,
    )
    cross_val_loader = create_loader(
        TokenDataset(token_store['val'][0], token_store['val'][1], token_labels['val']),
        batch_size=max(1, config.training.batch_size // 2),
        shuffle=False,
        num_workers=config.training.num_workers,
    )
    cross_result = train_model(
        model=cross_attention,
        train_loader=cross_train_loader,
        val_loader=cross_val_loader,
        device=runtime.device,
        epochs=config.training.epochs,
        learning_rate=config.training.learning_rate,
        weight_decay=config.training.weight_decay,
        log_dir=runtime.log_root / f"cross_attention_{CURRENT_STAGE}",
        checkpoint_path=runtime.checkpoint_root / f"cross_attention_{CURRENT_STAGE}.pt",
    )

In [ ]:
if RUN_CROSS_ATTENTION:
    cross_test_loader = create_loader(
        TokenDataset(token_store['test'][0], token_store['test'][1], token_labels['test']),
        batch_size=max(1, config.training.batch_size // 2),
        shuffle=False,
        num_workers=config.training.num_workers,
    )
    cross_metrics, cross_logits, cross_labels = evaluate_model(cross_attention.to(runtime.device), cross_test_loader, runtime.device)
    temperature = fit_temperature(cross_result.val_logits.to(runtime.device), cross_result.val_labels.to(runtime.device))
    calibrated_probs = torch.softmax(cross_logits / temperature, dim=1).cpu().numpy()
    calibration = expected_calibration_error(calibrated_probs, cross_labels.numpy())
    cross_predictions = cross_logits.argmax(dim=1).numpy()
    cross_confidence = torch.softmax(cross_logits, dim=1).max(dim=1).values.numpy()

    cross_view = split_frames['test'].copy()
    cross_view['pred_label'] = [CLASS_ORDER[idx] for idx in cross_predictions]
    cross_view['confidence'] = cross_confidence
    cross_view['correct'] = cross_view['label'] == cross_view['pred_label']

    cross_train_pdf = runtime.figure_root / f"cross_attention_training_{CURRENT_STAGE}.pdf"
    cross_train_png = runtime.figure_root / f"cross_attention_training_{CURRENT_STAGE}.png"
    cross_conf_pdf = runtime.figure_root / f"cross_attention_confusion_{CURRENT_STAGE}.pdf"
    cross_conf_png = runtime.figure_root / f"cross_attention_confusion_{CURRENT_STAGE}.png"
    cross_cal_pdf = runtime.figure_root / f"cross_attention_calibration_{CURRENT_STAGE}.pdf"
    cross_cal_png = runtime.figure_root / f"cross_attention_calibration_{CURRENT_STAGE}.png"

    for path in [cross_train_pdf, cross_train_png]:
        save_training_curves(cross_result.history, path, "Cross-Attention Training")
    for path in [cross_conf_pdf, cross_conf_png]:
        save_confusion_matrix(cross_metrics['confusion_matrix'], CLASS_ORDER, path, "Cross-Attention Confusion Matrix")
    for path in [cross_cal_pdf, cross_cal_png]:
        save_reliability_diagram(calibration.bin_centers, calibration.bin_accuracy, calibration.bin_confidence, path, "Cross-Attention Reliability")

    cross_metrics_path = runtime.metrics_root / f"cross_attention_metrics_{CURRENT_STAGE}.json"
    cross_metrics_path.write_text(json.dumps({**cross_metrics, 'temperature': float(temperature), 'ece': calibration.ece}, indent=2), encoding='utf-8')
    print("Cross-attention metrics", cross_metrics)
    print(f"Cross-attention calibration ECE: {calibration.ece:.4f}")

    show_saved_figure(cross_train_png, "Cross-Attention Training Curves")
    show_saved_figure(cross_conf_png, "Cross-Attention Confusion Matrix")
    show_saved_figure(cross_cal_png, "Cross-Attention Reliability Diagram")
    show_prediction_examples(cross_view.query('correct'), "Cross-Attention Correct Examples", n=3, seed=19)
    show_prediction_examples(cross_view.query('not correct'), "Cross-Attention Failure Examples", n=3, seed=23)

## 9. Qwen2.5-VL Zero-Shot Reference

This section runs only on the fixed subset and caches every raw output so reruns stay cheap.

In [ ]:
qwen_subset = sample_qwen_subset(benchmark_frame, subset_size=config.data.qwen_subset_size, seed=config.runtime.split_seed)
qwen_subset_path = runtime.benchmark_root / f"qwen_subset_{CURRENT_STAGE}.csv"
qwen_subset.to_csv(qwen_subset_path, index=False)
print(f"Saved Qwen subset to {qwen_subset_path}")
qwen_subset[['label', 'edit_family']].value_counts().head(12)

In [ ]:
if RUN_QWEN:
    qwen_bundle = load_qwen_bundle(config.model.qwen_name, use_4bit=config.model.use_qwen_4bit)
    qwen_outputs = run_qwen_inference(
        qwen_subset,
        qwen_bundle,
        output_dir=runtime.qwen_root / CURRENT_STAGE,
        max_new_tokens=config.model.max_qwen_tokens,
    )
    qwen_outputs['y_true'] = qwen_outputs['label'].map(CLASS_ORDER.index)
    qwen_outputs['y_pred'] = qwen_outputs['pred_label'].map(lambda label: CLASS_ORDER.index(label) if label in CLASS_ORDER else -1)
    qwen_eval = qwen_outputs.loc[qwen_outputs['y_pred'] >= 0].copy()
    qwen_metrics = compute_classification_metrics(qwen_eval['y_true'].to_numpy(), qwen_eval['y_pred'].to_numpy())
    (runtime.metrics_root / f"qwen_metrics_{CURRENT_STAGE}.json").write_text(json.dumps(qwen_metrics, indent=2), encoding='utf-8')
    print("Qwen metrics", qwen_metrics)

## 10. Qualitative and Per-Family Views

Use this section after model runs to export qualitative examples and summary plots.

In [ ]:
family_counts = benchmark_frame.groupby(['edit_family', 'label']).size().reset_index(name='count')
family_pivot = family_counts.pivot(index='edit_family', columns='label', values='count').fillna(0).sort_index()

family_pdf = runtime.figure_root / f"benchmark_family_counts_{CURRENT_STAGE}.pdf"
family_png = runtime.figure_root / f"benchmark_family_counts_{CURRENT_STAGE}.png"

for output_path in [family_pdf, family_png]:
    fig, ax = plt.subplots(figsize=(9, 5))
    family_pivot.plot(kind='bar', ax=ax)
    ax.set_title('Benchmark Distribution by Edit Family')
    ax.set_xlabel('Edit Family')
    ax.set_ylabel('Count')
    ax.legend(title='Label')
    fig.tight_layout()
    fig.savefig(output_path, bbox_inches='tight')
    plt.close(fig)

print(family_pivot)
show_saved_figure(family_png, 'Benchmark Distribution by Edit Family', figsize=(10, 5))

In [ ]:
qualitative_source = split_frames['test'].copy()
qualitative_pdf = runtime.figure_root / f"qualitative_panel_{CURRENT_STAGE}.pdf"
qualitative_png = runtime.figure_root / f"qualitative_panel_{CURRENT_STAGE}.png"
for output_path in [qualitative_pdf, qualitative_png]:
    save_qualitative_panel(qualitative_source, output_path, "Prototype Qualitative Examples")
print(f"Saved qualitative panel to {qualitative_pdf}")
show_saved_figure(qualitative_png, "Prototype Qualitative Panel", figsize=(12, 10))

## 11. Summary of Generated Artifacts

This section gives a quick view of the benchmark, metrics, checkpoints, and figure outputs produced by the notebook.

In [ ]:
artifact_roots = [
    runtime.benchmark_root,
    runtime.metrics_root,
    runtime.figure_root,
    runtime.checkpoint_root,
    runtime.qwen_root,
]
for root in artifact_roots:
    print(f"\nArtifacts under {root}")
    for path in sorted(root.glob('*'))[:10]:
        print(f"  - {path.name}")